# Mervis -- Phase 1: Fine-tune Phi-4-mini on Google Colab

Fine-tune **microsoft/Phi-4-mini-instruct** to answer as the two-headed robot
(**Mervin** the gloomy one, **Mervis** the cheerful one) using **QLoRA** so it
fits on a free Colab **T4 (16 GB)**.

**How to run:** `Runtime -> Run all`. When prompted, upload
`mervin_mervis_finetune.csv` (the dataset from this repo). At the end you get a
`mervis-merged.zip` containing the merged model for Phase 2.

> **GPU check:** make sure you picked a GPU runtime --
> `Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU`.

## 1. Install dependencies

Pinned to a known-compatible set so the notebook 'just runs'.

In [ ]:
%pip install -q \
  "transformers==4.49.0" \
  "trl==0.12.2" \
  "peft==0.14.0" \
  "accelerate==1.3.0" \
  "bitsandbytes==0.45.3" \
  "datasets==3.2.0" \
  "sentencepiece" \
  "tiktoken"

In [ ]:
# Sanity check: confirm a GPU is attached.
import torch
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), (
    "No GPU. Set Runtime -> Change runtime type -> T4 GPU, then Run all again."
)
print("GPU:", torch.cuda.get_device_name(0))

## 2. Get the dataset

Clone the public repo to pull `mervin_mervis_finetune.csv` (and this notebook's
sibling files) onto the Colab VM. Re-running is safe -- it skips the clone if the
repo is already present.

In [ ]:
import os
from datasets import load_dataset

REPO_DIR = "mervis"
if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 https://github.com/freeideas/mervis.git

CSV_PATH = os.path.join(REPO_DIR, "mervin_mervis_finetune.csv")
assert os.path.exists(CSV_PATH), f"CSV not found at {CSV_PATH}"

raw = load_dataset("csv", data_files=CSV_PATH, split="train")
print(raw)
print(raw[0])

## 3. Model + tokenizer (4-bit base)

Load Phi-4-mini in 4-bit (QLoRA) via bitsandbytes.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE_MODEL = "microsoft/Phi-4-mini-instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
    attn_implementation="eager",
)
model.config.use_cache = False
print("Loaded", BASE_MODEL)

## 4. Render each row into the Phi-4 chat template

We let the tokenizer build the `<|user|> ... <|assistant|> ...` string so it
exactly matches what Phi-4-mini expects. The full `response` (both `<Mervin>`
and `<Mervis>` tags) is the assistant turn.

In [ ]:
def to_text(example):
    messages = [
        {"role": "user", "content": example["prompt"]},
        {"role": "assistant", "content": example["response"]},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

dataset = raw.map(to_text, remove_columns=raw.column_names)
print(dataset[0]["text"])

## 5. LoRA config

In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

## 6. Train

~262 examples, 3 epochs. On a T4 this takes roughly 10-20 minutes.

In [ ]:
from trl import SFTTrainer, SFTConfig

ADAPTER_DIR = "mervis-lora"

sft_config = SFTConfig(
    output_dir=ADAPTER_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    fp16=True,
    max_seq_length=1024,
    dataset_text_field="text",
    packing=False,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
)

trainer.train()
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("Saved LoRA adapters to", ADAPTER_DIR)

## 7. Quick smoke test

Generate from the freshly trained adapters to confirm both personas appear.

In [ ]:
from transformers import pipeline

gen = pipeline("text-generation", model=trainer.model, tokenizer=tokenizer)
prompt = tokenizer.apply_chat_template(
    [{"role": "user", "content": "What is the capital of France?"}],
    tokenize=False, add_generation_prompt=True,
)
out = gen(prompt, max_new_tokens=200, do_sample=False)[0]["generated_text"]
print(out[len(prompt):])

## 8. Merge LoRA into the base weights

Reload the base model in fp16 (no quantization), apply the adapters, and merge.
Merging into a 4-bit model is not supported, so we reload fresh -- Phi-4-mini in
fp16 is ~7.6 GB, which fits on the T4.

In [ ]:
import gc, torch
from peft import PeftModel

# Free the 4-bit training model first.
del trainer, model
gc.collect()
torch.cuda.empty_cache()

MERGED_DIR = "mervis-merged"

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
merged = PeftModel.from_pretrained(base, ADAPTER_DIR)
merged = merged.merge_and_unload()
merged.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)
print("Merged model saved to", MERGED_DIR)

## 9. Download the merged model

Zip it and download for Phase 2 (browser conversion). The zip is a few GB, so
this can take a while -- alternatively, save straight to Google Drive (next
cell, commented out) or push to the Hugging Face Hub.

In [ ]:
import shutil
shutil.make_archive("mervis-merged", "zip", "mervis-merged")
print("Created mervis-merged.zip")

try:
    from google.colab import files  # type: ignore
    files.download("mervis-merged.zip")
except ImportError:
    print("Not in Colab -- find mervis-merged.zip in the working directory.")

In [ ]:
# --- Option B: save to Google Drive instead of downloading ---
# from google.colab import drive
# drive.mount("/content/drive")
# shutil.copytree("mervis-merged", "/content/drive/MyDrive/mervis-merged")

# --- Option C: push to the Hugging Face Hub ---
# from huggingface_hub import login
# login()  # paste a write token
# merged.push_to_hub("your-username/mervis-phi4-mini")
# tokenizer.push_to_hub("your-username/mervis-phi4-mini")